# Part II exercises — linear algebra with tensor networks

About 30 minutes. As before: predict, then measure.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_quimb as qq
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (8, 3), "axes.grid": True, "grid.alpha": 0.3})

---
## Exercise 1 — Crank–Nicolson, and what second order buys

**Point:** the solver is a component you can swap the scheme around, not the scheme
itself.

Backward Euler is $(\mathbb{1} - \alpha\Delta t L)u^{k+1} = u^k$ and is first order in
time. Crank–Nicolson,

$$\left(\mathbb{1} - \tfrac{\alpha\Delta t}{2}L\right)u^{k+1}
 = \left(\mathbb{1} + \tfrac{\alpha\Delta t}{2}L\right)u^{k},$$

is second order and still unconditionally stable. Build both, integrate to $T = 0.005$,
and measure the convergence order against the exact solution
$u(x,t) = \sin(2\pi x)\,e^{-4\pi^2 t}$.

In [ ]:
n = 12
N1 = 2 ** n
h = 2.0 ** -n
L = tn.qtt_laplacian(n, dx=h)
T = 0.005

# Compare against the *semi-discrete* exact solution, not the continuous one. The
# discrete Dirichlet Laplacian has known eigenpairs, so this reference is exact in space
# and the only error left is the time discretisation -- which is the thing being
# measured. Comparing against the continuous solution instead puts a spatial error floor
# under the plot and hides the second-order convergence completely.
j = 1
v0 = np.sin(np.pi * j * (np.arange(N1) + 1) / (N1 + 1))
lam = -4.0 / h ** 2 * np.sin(np.pi * j / (2 * (N1 + 1))) ** 2
exact = lambda t: v0 * np.exp(lam * t)


def integrate_backward_euler(dt):
    A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -dt)), 1e-13)
    u = tn.qtt_from_vector(exact(0.0), eps=1e-13)
    for _ in range(int(round(T / dt))):
        u = tn.dmrg_solve(A, u, x0=u, sweeps=2, eps=1e-12, chi_max=40)
    return u


def integrate_crank_nicolson(dt):
    # TODO: build A (left-hand side) and B (right-hand side) as MPOs, then loop:
    #   rhs = round(mpo_apply(B, u));  u = dmrg_solve(A, rhs, x0=u, ...)
    raise NotImplementedError


# for dt in (T / 2, T / 4, T / 8, T / 16):
#     err_be = ...
#     err_cn = ...
#     print(dt, err_be, err_cn)

Plot both errors against $\Delta t$ on log axes and read off the slopes. You should see
1 and 2. Then answer: **why is the cost per step essentially unchanged?**

---
## Exercise 2 — two-dimensional Poisson, and the rank you actually need

**Point:** the pressure solve of Part III, in miniature.

Solve $\nabla^2 p = f$ on a $2^m \times 2^m$ grid with a smooth right-hand side, using
`qq.laplacian_2d` and `tn.dmrg_solve`. Then:

* sweep `chi_max` from 4 to 64 and plot the residual;
* time the solve at $m = 5, 6, 7, 8$ and check whether the cost grows like $m$ (i.e.
  logarithmically in the number of cells) or like $4^m$.

In [ ]:
m = 6
N = 2 ** m
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
rhs_dense = np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)

# TODO (a): b = qq.from_grid(rhs_dense, ...) ; Lap = qq.laplacian_2d(m, h=1.0 / N)
# TODO (b): for chi in (4, 8, 16, 32, 64): solve and record tn.residual(...)
# TODO (c): time the chi=32 solve for m in (5, 6, 7, 8) and plot time against m

The $x$-axis of your timing plot is $m = \log_2 N$. If the curve is anything close to a
straight line in $m$, the cost is polylogarithmic in the number of grid cells, which is
the entire claim of the method.

---
## Exercise 3 — cross-interpolate something you cannot see

**Point:** TT-cross turns "I can evaluate $f$" into "I have $f$ in compressed form",
and the evaluation count is the figure of merit.

`mystery` below is a black box: it takes an array of binary multi-indices and returns
values. You may not look inside it, and you may not build the dense array.

1. Reconstruct it with `tn.tt_cross` at ranks 2, 4, 8, 16 and plot the reported number
   of evaluations against rank.
2. Decide, from the rank at which the reconstruction stops improving, roughly how
   complicated the function is.
3. Only then, plot it.

In [ ]:
_rng = np.random.default_rng(11)
_n = 18
_w = 2.0 ** -(np.arange(_n) + 1)
_calls = {"count": 0}


def mystery(idx):
    _calls["count"] += len(idx)
    t = idx @ _w
    return (np.exp(-3 * t) * np.cos(10 * np.pi * t) + 0.3 * np.sin(2 * np.pi * t))


# TODO: reconstruct, count evaluations, compare against 2**_n = 262,144.

---
## Exercise 4 — price the geometry

**Point:** $\alpha$ in the smoothed mask is a dial between fidelity and rank, and you
should be able to set it from a requirement rather than by taste.

For the disc of radius $0.25$ on a $512^2$ grid:

1. Confirm that the relative $L_2$ distance between the mask and the true indicator
   scales like $\alpha^{-1/2}$ (fit a slope on log-log axes).
2. Suppose the requirement is "geometry accurate to 5%". Find the smallest $\alpha$
   that meets it, and report the bond dimension needed to represent that mask to a
   relative tolerance of $10^{-2}$.
3. Compare with the bond dimension of the sharp indicator at the same tolerance. Is the
   smoothing buying you anything here? Be honest about the answer.

In [ ]:
mm = 9
Nm = 2 ** mm
xg = np.linspace(0, 1, Nm, endpoint=False)
Xg, Yg = np.meshgrid(xg, xg, indexing="ij")
q = 0.25 ** 2 - ((Xg - 0.5) ** 2 + (Yg - 0.5) ** 2)
indicator = (q > 0).astype(float)
smooth_mask = lambda a: 1 - np.exp(-a * (q + np.abs(q)))

# TODO

---
## Solutions

In [ ]:
# --- Exercise 1 ---
def integrate_cn(dt):
    A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -dt / 2)), 1e-13)
    B = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, dt / 2)), 1e-13)
    u = tn.qtt_from_vector(exact(0.0), eps=1e-13)
    for _ in range(int(round(T / dt))):
        rhs = tn.tt_round(tn.mpo_apply(B, u), eps=1e-12)
        u = tn.dmrg_solve(A, rhs, x0=u, sweeps=2, eps=1e-12, chi_max=40)
    return u


ref = exact(T)
dts, e_be, e_cn = [], [], []
for dt in (T / 2, T / 4, T / 8, T / 16):
    dts.append(dt)
    e_be.append(np.linalg.norm(tn.qtt_to_vector(integrate_backward_euler(dt)) - ref)
                / np.linalg.norm(ref))
    e_cn.append(np.linalg.norm(tn.qtt_to_vector(integrate_cn(dt)) - ref)
                / np.linalg.norm(ref))
dts = np.array(dts)
print(f"backward Euler slope: {np.polyfit(np.log(dts), np.log(e_be), 1)[0]:.2f}")
print(f"Crank-Nicolson slope: {np.polyfit(np.log(dts), np.log(e_cn), 1)[0]:.2f}")
plt.loglog(dts, e_be, "o-", label="backward Euler")
plt.loglog(dts, e_cn, "s-", label="Crank-Nicolson")
plt.xlabel("$\\Delta t$"), plt.ylabel("relative error"), plt.legend()
plt.tight_layout()
print("\nCost per step is unchanged because the extra work is one MPO application on the")
print("right-hand side -- O(n chi^2 chi_MPO^2) -- against a linear solve that costs")
print("O(n chi^4) or worse. The solve dominates either way.")

In [ ]:
# --- Exercise 2 ---
b = qq.from_grid(rhs_dense, eps=1e-12)
Lap = qq.laplacian_2d(m, h=1.0 / N)
print("chi_max sweep at m = 6:")
for chi in (4, 8, 16, 32, 64):
    p = tn.dmrg_solve(Lap, b, sweeps=3, eps=1e-10, chi_max=chi)
    print(f"  chi_max = {chi:>3}: residual {tn.residual(Lap, p, b):.2e}, "
          f"chi used {max(tn.tt_ranks(p))}")

print("\nscaling with resolution at chi_max = 32:")
ms, ts = [], []
for mi in (5, 6, 7, 8):
    Ni = 2 ** mi
    xi = np.linspace(0, 1, Ni, endpoint=False)
    Xi, Yi = np.meshgrid(xi, xi, indexing="ij")
    bi = qq.from_grid(np.sin(2 * np.pi * Xi) * np.sin(2 * np.pi * Yi), eps=1e-12)
    Li = qq.laplacian_2d(mi, h=1.0 / Ni)
    t0 = time.perf_counter()
    tn.dmrg_solve(Li, bi, sweeps=2, eps=1e-10, chi_max=32)
    ts.append(time.perf_counter() - t0)
    ms.append(mi)
    print(f"  m = {mi}: {Ni**2:>9,d} cells, {ts[-1]:.2f} s")
print(f"\ncells grew by {4**(ms[-1]-ms[0]):,d}x, runtime by {ts[-1]/ts[0]:.1f}x")

In [ ]:
# --- Exercise 3 ---
_calls["count"] = 0
xs18 = np.linspace(0, 1, 2 ** _n, endpoint=False)
truth = np.exp(-3 * xs18) * np.cos(10 * np.pi * xs18) + 0.3 * np.sin(2 * np.pi * xs18)
for rank in (2, 4, 8, 16):
    _calls["count"] = 0
    c = tn.tt_cross(mystery, [2] * _n, rank=rank, sweeps=3)
    err = np.linalg.norm(tn.qtt_to_vector(c) - truth) / np.linalg.norm(truth)
    print(f"rank {rank:>2}: {_calls['count']:>7,d} evaluations "
          f"({100 * _calls['count'] / 2**_n:>6.2f}% of the grid), rel err {err:.2e}")
print("\nThe error stops improving at rank 4: exp * cos is a sum of two complex")
print("exponentials (rank 2) and the extra sine adds 2 more. The function was never")
print("more complicated than four numbers per bond, and cross found that out for the")
print("price of a few thousand evaluations rather than a quarter of a million.")

In [ ]:
# --- Exercise 4 ---
alphas = np.array([25, 50, 100, 200, 400, 800, 1600, 3200])
dists = np.array([np.linalg.norm(smooth_mask(a) - indicator) / np.linalg.norm(indicator)
                  for a in alphas])
slope = np.polyfit(np.log(alphas), np.log(dists), 1)[0]
print(f"fitted exponent: {slope:.3f}   (theory: -0.5)")

target = 0.05
a_star = alphas[dists < target][0] if (dists < target).any() else None
print(f"smallest tabulated alpha meeting 5%: {a_star} (distance {dists[alphas == a_star][0]:.3f})")
chi_mask = max(tn.tt_ranks(qq.from_grid(smooth_mask(a_star), eps=1e-2)))
chi_sharp = max(tn.tt_ranks(qq.from_grid(indicator, eps=1e-2)))
print(f"bond dimension at 1e-2 tolerance: smoothed {chi_mask}, sharp {chi_sharp}")
print("""
Be honest about part 3: at 5% geometric accuracy the smoothed mask is barely cheaper
than the indicator, and at tight tolerance it is more expensive. The smoothing earns
its keep at loose accuracy and, more importantly, by being *differentiable* and by
giving the approximation an error bar. "Smoothing reduces rank" is folklore; "alpha
prices the trade between geometric fidelity and rank" is the defensible version.""")
plt.loglog(alphas, dists, "o-")
plt.loglog(alphas, dists[0] * (alphas / alphas[0]) ** -0.5, "k--", label=r"$\alpha^{-1/2}$")
plt.xlabel(r"$\alpha$"), plt.ylabel("relative $L_2$ distance to indicator"), plt.legend()
plt.tight_layout()